# Team 4_4_Graph_Feature

In [0]:
# Setup
!pip install networkx

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# imports
from graphframes import GraphFrame
from pyspark.sql.functions import col
import re
import heapq
import itertools
import numpy as np
import networkx as nx # 
from collections import defaultdict, deque
import matplotlib.pyplot as plt
%matplotlib inline

In [0]:

data_BASE_DIR = "dbfs:/mnt/mids-w261/"
# Team folder
section = "4"
number = "4"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
data_path_12M = f"{folder_path}/data_12M/"


df_1Y_features = spark.read.parquet(f"{data_path_12M}/df_joined_1Y_2015_features.parquet/")

print(f"Our feature dataset has {df_1Y_features.count()} rows and {len(df_1Y_features.columns)} columns.") 

# print the first 5 rows
display(df_1Y_features.limit(5))


Our feature dataset has 14536464 rows and 47 columns.


ORIGIN,MONTH,flight_id,prediction_utc,origin_obs_utc,asof_minutes,FL_DATE,YEAR,QUARTER,DAY_OF_MONTH,DAY_OF_WEEK,OP_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,CRS_DEP_TIME,DEST,DISTANCE,DISTANCE_GROUP,HourlyDryBulbTemperature,HourlyDewPointTemperature,HourlyWetBulbTemperature,HourlyPrecipitation,HourlyWindSpeed,HourlyWindDirection,HourlyVisibility,HourlyRelativeHumidity,HourlyStationPressure,HourlySeaLevelPressure,HourlyAltimeterSetting,HourlySkyConditions,ORIGIN_TYPE,DEST_TYPE,DEP_DEL15,DEP_DELAY,DEP_HOUR,DEP_MINUTE,dep_time_bucket,is_weekend,peak_travel_month,peak_travel_hour,is_holiday,dep_delay_24h_rolling_avg_by_origin,dep_delay15_24h_rolling_avg_by_origin,dep_delay_24h_rolling_avg_by_origin_carrier,dep_delay15_24h_rolling_avg_by_origin_carrier,dep_delay_24h_rolling_avg_by_origin_dayofweek,dep_delay15_24h_rolling_avg_by_origin_dayofweek
AKN,6,2019-06-10|AS|163|AKN|ANC,2019-06-10T23:50:00Z,2019-06-10T22:54:00Z,56,2019-06-10,2019,2,10,1,AS,163,N403AS,1750,ANC,288.0,2,54.0,45.0,49.0,0.0,3.0,240.0,10.0,72.0,29.84000015258789,29.8799991607666,29.889999389648438,41.0,medium_airport,large_airport,0.0,2.0,17,50,evening,0,1,1,0,0.0,0.0,0.0,0.0,-3.6,0.016666666666666666
AKN,6,2019-06-10|AS|163|AKN|ANC,2019-06-10T23:50:00Z,2019-06-10T22:54:00Z,56,2019-06-10,2019,2,10,1,AS,163,N403AS,1750,ANC,288.0,2,54.0,45.0,49.0,0.0,3.0,240.0,10.0,72.0,29.84000015258789,29.8799991607666,29.889999389648438,41.0,medium_airport,large_airport,0.0,2.0,17,50,evening,0,1,1,0,0.0,0.0,0.0,0.0,-3.6,0.016666666666666666
AKN,6,2019-06-17|AS|163|AKN|ANC,2019-06-17T23:50:00Z,2019-06-17T22:54:00Z,56,2019-06-17,2019,2,17,1,AS,163,N565AS,1750,ANC,288.0,2,55.0,50.0,52.0,0.0,8.0,200.0,10.0,83.0,29.639999389648438,29.690000534057617,29.690000534057617,41.0,medium_airport,large_airport,0.0,-7.0,17,50,evening,0,1,1,0,-6.0,0.0,-6.0,0.0,-3.6,0.016666666666666666
AKN,6,2019-06-17|AS|163|AKN|ANC,2019-06-17T23:50:00Z,2019-06-17T22:54:00Z,56,2019-06-17,2019,2,17,1,AS,163,N565AS,1750,ANC,288.0,2,55.0,50.0,52.0,0.0,8.0,200.0,10.0,83.0,29.639999389648438,29.690000534057617,29.690000534057617,41.0,medium_airport,large_airport,0.0,-7.0,17,50,evening,0,1,1,0,-6.0,0.0,-6.0,0.0,-3.6,0.016666666666666666
AKN,6,2019-06-24|AS|163|AKN|ANC,2019-06-24T23:50:00Z,2019-06-24T22:54:00Z,56,2019-06-24,2019,2,24,1,AS,163,N517AS,1750,ANC,288.0,2,53.0,50.0,51.0,0.0,7.0,260.0,10.0,89.0,29.959999084472656,30.010000228881836,30.010000228881836,41.0,medium_airport,large_airport,0.0,-11.0,17,50,evening,0,1,1,0,-1.0,0.0,-1.0,0.0,-3.6,0.016666666666666666


We will design our graph database with airports as the nodes and the flights as the edges.  The number of flights originating from an airport will be its attribute. The number of flights reaching an airport will also be its attributes. 

In [0]:
edges = (
    df_1Y_features
    .groupBy("ORIGIN", "DEST")
    .count()
    .withColumnRenamed("ORIGIN", "src")
    .withColumnRenamed("DEST", "dst")
    .withColumnRenamed("count", "weight")
)


# count of number of edges
print(f"There are {edges.count()} edges in our graph.")

#display edges sorted by distance
display(edges.orderBy(col("weight").desc()), limit=10)

There are 6515 edges in our graph.


src,dst,weight
LGA,ORD,28338
ORD,LGA,28216
LAX,SFO,28176
SFO,LAX,28172
LAX,JFK,25502
JFK,LAX,25396
LAX,LAS,23234
LAS,LAX,23224
HNL,OGG,21442
OGG,HNL,21434


In [0]:
vertices = df_1Y_features.select("ORIGIN").distinct().withColumnRenamed("ORIGIN", "id").union(df_1Y_features.select("DEST").distinct().withColumnRenamed("DEST", "id"))

# count of number of nodes
print(f"There are {vertices.count()} nodes in our graph.")

display(vertices, limit=10)


There are 720 nodes in our graph.


id
BKG
BWI
TPA
APN
CMX
PRC
OKC
ROA
BRW
OTH


In [0]:


## Let's display the graph vertices - GCP Solution, for Databricks, just display(g.vertices)
g = GraphFrame(vertices, edges)
display(g.vertices.limit(10).toPandas())

## Let's display the graph edges sorted in descending order by weight
display(g.edges.orderBy(col("weight").desc()).limit(10).toPandas())

## Let's display inDegrees sorted in descending order
display(g.inDegrees.orderBy(col("inDegree").desc()).limit(10).toPandas())


## Let's display outDegrees sorted in descending order
display(g.outDegrees.orderBy(col("outDegree").desc()).limit(10).toPandas())



id
MSY
GEG
DRT
SNA
BUR
GRR
JLN
EUG
MYR
PVD


src,dst,weight
LGA,ORD,28338
ORD,LGA,28216
LAX,SFO,28176
SFO,LAX,28172
LAX,JFK,25502
JFK,LAX,25396
LAX,LAS,23234
LAS,LAX,23224
HNL,OGG,21442
OGG,HNL,21434


id,inDegree
ORD,187
DFW,183
DEN,177
ATL,168
CLT,136
MSP,130
IAH,120
DTW,116
LAS,114
LAX,110


id,outDegree
ORD,186
DFW,184
DEN,175
ATL,168
CLT,136
MSP,130
IAH,119
DTW,116
LAS,114
LAX,109


In [0]:
# Calculate pagerank for each node
pr = g.pageRank(resetProbability=0.15, maxIter=10)
display(pr.vertices.orderBy(col("pagerank").desc()).limit(10).toPandas())
# Calculate degree centrality for each node
dc = g.degrees
display(dc.orderBy(col("degree").desc()).limit(10).toPandas())


id,pagerank
DFW,11.270068461306332
DFW,11.270068461306332
ORD,10.800858701476958
ORD,10.800858701476958
DEN,10.560525947764576
DEN,10.560525947764576
ATL,8.936903999115492
ATL,8.936903999115492
MSP,7.240196442696727
MSP,7.240196442696727


id,degree
ORD,373
DFW,367
DEN,352
ATL,336
CLT,272
MSP,260
IAH,239
DTW,232
LAS,228
LAX,219


In [0]:
# add pagerank to the dataframe df_1Y_features
df_with_pagerank = df_1Y_features.join(pr.vertices, df_1Y_features.ORIGIN == pr.vertices.id, 'left').drop('id')
# add degree centrality to the dataframe df_1Y_features
df_pr_dc = df_with_pagerank.join(dc, df_1Y_features.ORIGIN == dc.id, 'left').drop('id')






#### Betweeness

In [0]:
edges = (
    df_1Y_features
    .groupBy("ORIGIN", "DEST")
    .count()
    .withColumnRenamed("ORIGIN", "src")
    .withColumnRenamed("DEST", "dst")
    .withColumnRenamed("distance", "weight")
)


# count of number of edges
print(f"There are {edges.count()} edges in our graph.")

nodes = df_1Y_features.select("ORIGIN").distinct().withColumnRenamed("ORIGIN", "id").union(df_1Y_features.select("DEST").distinct().withColumnRenamed("DEST", "id"))



There are 6515 edges in our graph.


In [0]:
import networkx as nx
import pandas as pd
import pyspark.sql.functions as F

# Convert PySpark DataFrame to Pandas (collect to driver)
edges_pd = df_1Y_features.groupBy("ORIGIN", "DEST").agg(
    F.avg("DISTANCE").alias("distance"),
    F.count("*").alias("num_flights")
).toPandas()

# Create directed graph
G = nx.DiGraph()

# Add edges with distance as weight
for _, row in edges_pd.iterrows():
    G.add_edge(
        row['ORIGIN'], 
        row['DEST'], 
        weight=row['distance'],
        num_flights=row['num_flights']
    )

# Calculate betweenness centrality
# weight parameter uses edge weights for shortest path calculation
betweenness = nx.betweenness_centrality(G, weight='weight')

# Convert back to DataFrame
betweenness_df = pd.DataFrame([
    {'airport': k, 'betweenness': v} 
    for k, v in betweenness.items()
])

# Convert to Spark DataFrame and join back
betweenness_spark = spark.createDataFrame(betweenness_df)



In [0]:
df_with_pr_dc_betweenness = df_pr_dc.join(
    betweenness_spark,
    df_pr_dc.ORIGIN == betweenness_spark.airport,
    'left'
).drop('airport')

# handle nulls
pagerank_median = df_with_pr_dc_betweenness.agg(F.expr("percentile_approx(pagerank, 0.5)")).first()[0]
betweenness_median = df_with_pr_dc_betweenness.agg(F.expr("percentile_approx(betweenness, 0.5)")).first()[0]
degree_median = df_with_pr_dc_betweenness.agg(F.expr("percentile_approx(degree, 0.5)")).first()[0]

df_with_pr_dc_betweenness = df_with_pr_dc_betweenness.fillna({
    "pagerank": pagerank_median,
    "betweenness": betweenness_median,
    "degree": degree_median
})


#display number of rows and columns in the new dataframe
print(f"There are {df_with_pr_dc_betweenness.count()} rows and {len(df_with_pr_dc_betweenness.columns)} columns in the new dataframe.")
# display the number of rows and columns in the original dataframe
print(f"There are {df_1Y_features.count()} rows and {len(df_1Y_features.columns)} columns in the original dataframe.")

display(df_with_pr_dc_betweenness, limit=10)



There are 29072928 rows and 50 columns in the new dataframe.
There are 14536464 rows and 47 columns in the original dataframe.


ORIGIN,MONTH,flight_id,prediction_utc,origin_obs_utc,asof_minutes,FL_DATE,YEAR,QUARTER,DAY_OF_MONTH,DAY_OF_WEEK,OP_CARRIER,OP_CARRIER_FL_NUM,TAIL_NUM,CRS_DEP_TIME,DEST,DISTANCE,DISTANCE_GROUP,HourlyDryBulbTemperature,HourlyDewPointTemperature,HourlyWetBulbTemperature,HourlyPrecipitation,HourlyWindSpeed,HourlyWindDirection,HourlyVisibility,HourlyRelativeHumidity,HourlyStationPressure,HourlySeaLevelPressure,HourlyAltimeterSetting,HourlySkyConditions,ORIGIN_TYPE,DEST_TYPE,DEP_DEL15,DEP_DELAY,DEP_HOUR,DEP_MINUTE,dep_time_bucket,is_weekend,peak_travel_month,peak_travel_hour,is_holiday,dep_delay_24h_rolling_avg_by_origin,dep_delay15_24h_rolling_avg_by_origin,dep_delay_24h_rolling_avg_by_origin_carrier,dep_delay15_24h_rolling_avg_by_origin_carrier,dep_delay_24h_rolling_avg_by_origin_dayofweek,dep_delay15_24h_rolling_avg_by_origin_dayofweek,pagerank,degree,betweenness
ATL,1,2019-01-07|AA|1828|ATL|PHL,2019-01-07T08:21:00Z,2019-01-07T07:52:00Z,29,2019-01-07,2019,1,7,1,AA,1828,N680AW,521,PHL,666.0,3,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-7.0,5,21,morning,0,0,0,0,2.6134538152610443,0.09136546184738956,-1.6206896551724137,0.06896551724137931,8.756766002639768,0.16754871700608373,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|AA|1828|ATL|PHL,2019-01-07T08:21:00Z,2019-01-07T07:52:00Z,29,2019-01-07,2019,1,7,1,AA,1828,N680AW,521,PHL,666.0,3,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-7.0,5,21,morning,0,0,0,0,2.6134538152610443,0.09136546184738956,-1.6206896551724137,0.06896551724137931,8.756766002639768,0.16754871700608373,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|AA|1994|ATL|CLT,2019-01-07T08:39:00Z,2019-01-07T07:52:00Z,47,2019-01-07,2019,1,7,1,AA,1994,N102UW,539,CLT,226.0,1,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-8.0,5,39,morning,0,0,0,0,2.620100502512563,0.0914572864321608,-1.5357142857142858,0.07142857142857142,-7.0,0.0,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|AA|1994|ATL|CLT,2019-01-07T08:39:00Z,2019-01-07T07:52:00Z,47,2019-01-07,2019,1,7,1,AA,1994,N102UW,539,CLT,226.0,1,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-8.0,5,39,morning,0,0,0,0,2.620100502512563,0.0914572864321608,-1.5357142857142858,0.07142857142857142,-7.0,0.0,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|B6|996|ATL|BOS,2019-01-07T08:45:00Z,2019-01-07T07:52:00Z,53,2019-01-07,2019,1,7,1,B6,996,N516JB,545,BOS,946.0,4,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-7.0,5,45,morning,0,0,0,0,2.609437751004016,0.09136546184738956,-1.0,0.2,-7.5,0.0,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|B6|996|ATL|BOS,2019-01-07T08:45:00Z,2019-01-07T07:52:00Z,53,2019-01-07,2019,1,7,1,B6,996,N516JB,545,BOS,946.0,4,45.0,39.0,42.0,0.0,3.0,130.0,10.0,80.0,29.149999618530273,30.25,30.25,15.0,large_airport,large_airport,0.0,-7.0,5,45,morning,0,0,0,0,2.609437751004016,0.09136546184738956,-1.0,0.2,-7.5,0.0,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|AA|1151|ATL|MIA,2019-01-07T08:56:00Z,2019-01-07T08:52:00Z,4,2019-01-07,2019,1,7,1,AA,1151,N817NN,556,MIA,594.0,3,50.0,42.0,46.0,0.0,5.0,100.0,10.0,74.0,29.15999984741211,30.270000457763672,30.270000457763672,15.0,large_airport,large_airport,0.0,-8.0,5,56,morning,0,0,0,0,2.6231155778894473,0.0914572864321608,-1.7586206896551724,0.06896551724137931,-7.333333333333333,0.0,8.936903999115492,336,0.10524683193020144
ATL,1,2019-01-07|AA|1151|ATL|MIA,2019-01-07T08:56:00Z,2019-01-07T08:52:00Z,4,2019-01-07,2019,1,7,1,AA,1151,N817NN,556,MIA,594.0,3,50.0,42.0,46.0,0.0,5.0,100.0,10.0,74.0,29.15999984741211,30.270000457763672,30.270000457763672,15.0,large_airport,large_airport,0.0,-8.0,5,56,morning,0,0,0,0,2.6231155778894473,0.0914572864321608,-1.7586206896551724,

In [0]:
# write to parquet file
df_with_pr_dc_betweenness.write.mode("overwrite").parquet(f"{data_path_12M}df_joined_1Y_features_plus_gf.parquet")
